#  ⭐ `dim_soc`


In [2]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, build_row_hash

In [3]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [4]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [5]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17


# 🥈 Silver

normalized data, medium quality


In [6]:
silver = bronze.copy()

## hist_dim_soc

In [15]:
hist = (
    silver.rename(columns={"REACAO_EVTO_ADVERSO_MEDDRA_LLT": "REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR","SOC":"SOC_VALOR"})
    [["REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR", "PT", "HLT", "HLGT", "SOC_VALOR"]]
    .drop_duplicates()
    .sort_values(by=["SOC_VALOR", "HLGT", "HLT", "PT", "REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR"])
)
hist["LLT_CHAVE"] = 1 + np.arange(len(hist))
hist = hist.reset_index(drop=True)
hist = hist[["SOC_VALOR", "HLGT", "HLT", "PT", "REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR", "LLT_CHAVE"]]

In [16]:
hist.SOC_VALOR.nunique()

27

In [17]:
hist.SOC.value_counts()

AttributeError: 'DataFrame' object has no attribute 'SOC'

In [18]:
# RELACAO_MEDICAMENTO_EVENTO
soc_map = {
    "Infecções e infestações": 1,
    "Neoplasias benignas, malignas e não especificadas (incl. cistos e pólipos)": 2,
    "Distúrbios dos sistemas hematológico e linfático": 3,
    "Distúrbios do sistema imunitário": 4,
    "Distúrbios endócrinos": 5,
    "Distúrbios metabólicos e nutricionais": 6,
    "Distúrbios psiquiátricos": 7,
    "Distúrbios do sistema nervoso": 8,
    "Distúrbios oculares": 9,
    "Distúrbios do ouvido e do labirinto": 10,
    "Distúrbios cardíacos": 11,
    "Distúrbios vasculares": 12,
    "Distúrbios respiratórios, torácicos e do mediastino": 13,
    "Distúrbios gastrointestinais": 14,
    "Distúrbios hepatobiliares": 15,
    "Distúrbios dos tecidos cutâneos e subcutâneos": 16,
    "Distúrbios musculoesqueléticos e do tecido conjuntivo": 17,
    "Distúrbios renais e urinários": 18,
    "Quadros clínicos na gravidez, no puerpério e perinatais": 19,
    "Distúrbios do sistema reprodutor e da mama": 20,
    "Distúrbios congênitos, de família e genéticos": 21,
    "Distúrbios gerais e quadros clínicos no local de administração": 22,
    "investigações": 23,
    "Lesões, intoxicações e complicações de procedimentos": 24,
    "Procedimentos cirúrgicos e médicos": 25,
    "Circunstâncias sociais": 26,
    "Problemas relacionados ao produto": 27,
}
hist['SOC_CHAVE'] = hist['SOC_VALOR'].map(soc_map).astype('Int64') 
cols = ['SOC_CHAVE', 'SOC_VALOR', 'HLGT', 'HLT', 'PT','LLT_CHAVE', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR']
hist = hist[cols]
hist.head()

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [19]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_soc_llt/hist_dim_soc_llt.parquet", index=False)

# 🥇 Gold



## dim_atc

In [21]:
dim = hist.copy()
dim.to_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet", index=False)